<a href="https://colab.research.google.com/drive/1stGK2iPHUn_MM8xPtA9IlMifENSotXwH?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

<div style="text-align:center;">
<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/CuratorKIT/refs/heads/main/docs/assets/logo.png" width="450"></a>

<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/docs.png" width="150"></a>

</div>

<i>Star us on <a href="https://github.com/Lexsi-Labs/CuratorKIT">Github</a> </i>

To run this, press "*Runtime*" and press "*Run all*" on a Google Colab instance!

<div style="text-align:center;">
<a href="https://arxiv.org/abs/2606.21631"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/lexsilogo.png" width="200"></a>
</div>
</center>




# Synthetic generation — DPO

**Exporter:** `dpo` &nbsp;|&nbsp; **Gates:** HallucinationGate + RewardGate (always active)

Converts a PDF into DPO preference pairs (chosen vs rejected). Choose your generation task below.

### Valid tasks for DPO export

| Task | What it does | Rejected answer |
|------|-------------|----------------|
| `preference` | Generate chosen + quality-degraded rejected pair | Vague, shallow, or incomplete |
| `adversarial_preference` | Generate chosen + adversarially corrupted rejected | Contradicts source, parametric drift, domain mismatch |

Both produce `task_type="preference"` → exported as `dpo.jsonl`.

### Pipeline
```
PDF → chunk → dedup → clean → [TASK] → HallucinationGate → RewardGate → DiversityGate → dpo
```

**Key difference for DPO:** RewardGate **dual-scores** both chosen and rejected.
The pair passes only if `chosen >= threshold` AND `rejected < threshold` — enforcing quality contrast.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation,embedding,hf,parquet]" -q

print(f"CuratorKIT installed from {repo_dir}")



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ── PDF support (MinerU) ──────────────────────────────────────────────────
# Colab auto-detects GPU/CPU — no separate torch install needed.
# Model weights (~1 GB) download automatically on the first PDF parse.
!pip install -e "{repo_dir}[pdf]" -q

## 1. Configuration — pick your task

In [2]:
# Demo PDF
import urllib.request
urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "attention_is_all_you_need.pdf")
print("Downloaded.")

Downloaded.


## LLM Backend Setup

Configure your LLM endpoint before running. CuratorKIT uses [LiteLLM](https://docs.litellm.ai) under the hood, so any OpenAI-compatible endpoint works.

### Option A — vLLM (recommended for speed)

```bash
# Serve a local model with vLLM
pip install vllm
vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000
```

Then set in the config cell below:
```python
MODEL    = "openai/Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
```

### Option B — Ollama (easiest local setup)

```bash
# Install Ollama: https://ollama.com
ollama pull qwen2.5:7b
ollama serve  # starts on port 11434 by default
```

Then set:
```python
MODEL    = "ollama/qwen2.5:7b"
API_BASE = ""  # Ollama backend is auto-detected from the model prefix
```

### Option C — Any OpenAI-compatible API

```python
MODEL    = "openai/your-model-name"
API_BASE = "https://your-api-endpoint/v1"
# Set API key via env var: export OPENAI_API_KEY=your-key
# Or pass directly: llm_api_key="your-key" in CuratorConfig
```


In [3]:
MODEL    = "openai/Qwen/Qwen3-8B"
JUDGE    = "openai/Qwen/Qwen3-8B"
API_BASE = " "

In [4]:
from pathlib import Path
from curatorkit import Curator, CuratorConfig

# ═══════════════════════════════════════════════════════════════════════════
# Pick your task: "preference" | "adversarial_preference"
# ═══════════════════════════════════════════════════════════════════════════
TASK = "adversarial_preference"

# ── PDF source ────────────────────────────────────────────────────────────
PDF_PATH = "attention_is_all_you_need.pdf"  # ← or replace with your own PDF


# ── Pipeline params ───────────────────────────────────────────────────────
MAX_CHUNKS  = 30
CONCURRENCY = 32
OUTPUT_DIR  = Path(f"output/dpo_{TASK}")

print(f"Task   : {TASK}")
print(f"PDF    : {PDF_PATH}")
print(f"Model  : {MODEL}")
print(f"Output : {OUTPUT_DIR}/")

Task   : adversarial_preference
PDF    : attention_is_all_you_need.pdf
Model  : openai/Qwen/Qwen3-8B
Output : output/dpo_adversarial_preference/


## 2. Build config per task

In [6]:
api_base = API_BASE if API_BASE else None

# ── Shared config across all DPO tasks ────────────────────────────────────
shared = dict(
    dataset={"name": PDF_PATH, "max_samples": MAX_CHUNKS},
    pdf_chunk_strategy="heading",
    pdf_chunk_max_tokens=512,
    pdf_chunk_overlap_tokens=50,
    dedup="minhash",
    minhash_threshold=0.85,
    clean=True,
    llm_model=MODEL,
    llm_api_base=api_base,
    llm_max_tokens=2048,
    llm_concurrency=CONCURRENCY,
    generation_concurrency=CONCURRENCY,
    llm_extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    judge_llm_model=JUDGE,
    judge_llm_api_base=api_base,
    judge_llm_temperature=0.1,
    judge_concurrency=CONCURRENCY,
    judge_llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    hallucination_threshold=0.7,
    reward_threshold=0.75,
    reward_dimensions=["helpfulness", "honesty", "instruction_following", "depth"],
    diversity_threshold=0.92,
    embedding_device="cuda",
    embedding_batch_size=256,
    export_formats=["dpo"],
    output_dir=OUTPUT_DIR,
)

# ── Task-specific config ──────────────────────────────────────────────────
if TASK == "preference":
    task_config = dict(
        generation_task="preference",
        preference_mode="single_call",   # or "two_pass" for stronger degradation
        num_questions=1,
    )

elif TASK == "adversarial_preference":
    task_config = dict(
        generation_task="adversarial_preference",
        num_questions=1,
        # Fraction of pairs to inject with adversarial rejected answers.
        # Higher values → more adversarial pairs but also more gate rejections.
        # 0.5 is a balanced default; increase for more contrastive training data.
        injection_rate=0.5,
        injection_types=["contradicts_source", "parametric_drift", "domain_mismatch"],
    )

else:
    raise ValueError(f"Unknown TASK: {TASK}. Choose: preference | adversarial_preference")

config = CuratorConfig(**shared, **task_config, llm_api_key=" ")
print(f"Generation task: {config.generation_task}")
print(f"Export format  : {config.export_formats}")
print("\nRunning...")
result = Curator(config).run()
result.print_summary()

Generation task: adversarial_preference
Export format  : ['dpo']

Running...


2026-06-12 05:51:53.851 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:209 - Pipeline processing-window multi-file run. doc_count=1, total_pages=15, window_size=64, total_batches=1
2026-06-12 05:51:53.852 | DEBUG    | mineru.utils.pdf_image_tools:_load_images_from_pdf_bytes_range:289 - PDF image rendering uses 1 processes for pages 1-15: [(0, 14)]
2026-06-12 05:51:53.854 | DEBUG    | mineru.utils.pdf_image_tools:_create_pdf_render_executor:161 - PDF image rendering switches multiprocessing start method from fork to spawn
2026-06-12 05:51:53.858 | DEBUG    | mineru.utils.pdf_image_tools:_get_pdf_render_executor:218 - Created persistent PDF render executor with max_workers=2
2026-06-12 05:51:56.566 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:264 - Pipeline processing window batch 1/1: 15/15 pages, batch_pages=15, doc_slices=doc0:1-15
2026-06-12 05:51:56.569 | INFO     | mineru.backend.pipeline.pipeline_analyze:batch_image_analy

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

models/OCR/paddleocr_torch/en_PP-OCRv5_r(…):   0%|          | 0.00/23.9M [00:00<?, ?B/s]

OCR-det en: 100%|██████████| 30/30 [00:28<00:00,  1.06it/s]
2026-06-12 05:53:47.495 | DEBUG    | mineru.utils.model_utils:clean_vram:214 - gc time: 0.55
Seal Predict: 0it [00:00, ?it/s]
Processing pages: 100%|██████████| 15/15 [00:02<00:00,  6.09it/s]
2026-06-12 05:53:50.474 | DEBUG    | mineru.backend.pipeline.pipeline_analyze:doc_analyze_streaming:320 - processing-window multi-file infer finished, cost: 116.62, speed: 0.129 page/s
[AdversarialPreferenceTask] generating: 100%|██████████| 21/21 [01:34<00:00,  4.50s/sample]

────────────────────────────────────────────
  passed   :        0
  rejected :       25
  time     :   222.6s
  output   : output/dpo_adversarial_preference
────────────────────────────────────────────


## 3. Inspect

In [8]:
from collections import Counter

if result.passed:
    print(f"Passed: {len(result.passed):,}  |  Rejected: {len(result.rejected):,}\n")
    for i, s in enumerate(result.passed[:2]):
        print(f"── DPO Pair {i+1} ──")
        print(f"  Q:        {s.instruction[:100]}")
        print(f"  chosen:   {s.chosen[:100]}")
        print(f"  rejected: {s.rejected[:100]}")
        print(f"  injection: {s.metadata.get('injection_type', 'none')}")
        for rec in s.provenance_chain:
            if rec.step_name == "HallucinationGate":
                print(f"  grounding: {rec.notes.get('grounding_score', '-')}")
            if rec.step_name == "RewardGate":
                c = rec.notes.get('chosen_score', '-')
                r = rec.notes.get('rejected_score', '-')
                print(f"  chosen_score={c}  rejected_score={r}")
        print()

if result.rejected:
    reasons = Counter(r.rejection_reason for r in result.rejected)
    print("Rejection reasons:")
    for reason, count in reasons.most_common():
        print(f"  {count:>4}  {reason}")

Rejection reasons:
    21  generation_parse_failed:AdversarialPreferenceTask
     4  table_stubbed_phase_0


## 4. Task comparison

| Task | Rejected style | Detection difficulty | Best for |
|------|---------------|---------------------|----------|
| `preference` | Quality-degraded (vague, shallow) | Harder to detect — subtle quality drop | Generic DPO training |
| `adversarial_preference` | Adversarially corrupted (contradictions, drift) | Easier to detect — factual errors | Teaching model to reject specific failure modes |

### Understanding `rejected_above_threshold`
The most common DPO rejection. The adversarial injection was too subtle — the corrupted
answer still scored >= 0.75. The quality contrast between chosen and rejected is too small
for DPO training to learn from. Fixes:
- Switch to `preference_mode="two_pass"` (stronger degradation in rejected)
- Increase `injection_rate` to produce more adversarial pairs
- Use more detectable injection types (contradicts_source > domain_mismatch)